# Restructuration TF-IDF — fil conducteur

Vous reprenez votre classifieur d'intentions du **Module 1** et le
restructurez dans l'architecture du **Module 4**. **Complétez d'abord les TODO de `src/`**, puis ce notebook orchestre les
modules de `src/` ; il ne contient pas la logique, il l'appelle.


In [ ]:
# Amorçage : se placer à la racine du projet (dossier contenant src/)
import os, sys
while not os.path.isdir('src'):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        break
    os.chdir(parent)
sys.path.insert(0, os.getcwd())
print('cwd =', os.getcwd())

## Étape 1 — Charger les données

On regarde ce que produit le chargement : quelles colonnes a-t-on sous la main ?

In [ ]:
from src import data_loading, config
df = data_loading.load()
print(df.columns.tolist())
df.head()

## Étape 2 — La configuration centralisée

Tous les réglages vivent dans `config.py` : quelle colonne est le texte (l'entrée du modèle), laquelle est la cible, la graine, le découpage. Plus aucune valeur codée en dur ailleurs — on sait où regarder.

In [ ]:
print('Texte  :', config.TEXT_COL)
print('Cible  :', config.LABEL_COL)
print('Graine :', config.RANDOM_SEED)
print('Test   :', config.TEST_SIZE)

## Étape 3 — Découpage stratifié + TF-IDF

Découpage stratifié sur l'intention (préserve les classes rares). Le TF-IDF sera ajusté sur le train uniquement — c'est le Pipeline qui le garantit.

In [ ]:
from src import features
X_train, X_test, y_train, y_test = features.make_split(df)
print('train:', len(X_train), '| test:', len(X_test))
print('classes:', y_train.nunique())

## Étape 4 — Les candidats

Au Module 1 : un seul modèle. Ici, au moins trois familles, comparées.

In [ ]:
from src import models
candidates = models.build_candidates()
list(candidates.keys())

## Étape 5 — Les métriques

Multi-classe déséquilibré : on regarde le macro-F1 et la précision balancée, pas seulement l'exactitude.

In [ ]:
from src import evaluation
pipe = candidates['logreg'].fit(X_train, y_train)
y_pred = pipe.predict(X_test)
evaluation.evaluate(y_test, y_pred)

## Étape 6 — Benchmark complet

Le geste du Module 4 : tout le monde sur la même grille, résultats dans `outputs/results.csv`.

In [ ]:
from src.benchmark import run
results = run()
results

## Étape 7 — Lire le déséquilibre

Le rappel par classe révèle ce que l'exactitude globale masque : les intentions rares sont-elles ratées ?

In [ ]:
rep = evaluation.per_class_report(y_test, y_pred)
rep[['recall','support']].sort_values('support').round(2)

## Étape 8 — Conclure

En une cellule markdown : quel modèle retenez-vous, sur quelle métrique, et quel défaut de dette technique avez-vous corrigé au passage (l'ajustement du TF-IDF sur le train uniquement) ?